# 02 - Data Preparation & Cost Engineering
**Project:** Energy Consumption & Business Performance Analytics  
**Objective:** Execute standardized preprocessing, join relational dimensions, engineer utility cost metrics, and perform Price vs. Volume variance decomposition.


In [ ]:
import os
import pandas as pd
import numpy as np

raw_excel_path = os.path.join("..", "data", "raw", "Energy Consumption Data.xlsx")
df_ec = pd.read_excel(raw_excel_path, sheet_name="Energy Consumptions")
df_rates = pd.read_excel(raw_excel_path, sheet_name="Rates")
df_bm = pd.read_excel(raw_excel_path, sheet_name="Building Master")


## 1. Standardization & Calendar Feature Engineering


In [ ]:
df = df_ec.copy()
df['Date'] = pd.to_datetime(df['Date'])
df['Building'] = df['Building'].astype(str).str.strip()
df['Water_Consumption'] = df['Water Consumption'].astype(np.int64)
df['Electricity_Consumption'] = df['Electricity Consumption'].astype(np.int64)
df['Gas_Consumption'] = df['Gas Consumption'].astype(np.int64)
df = df.drop(columns=['Water Consumption', 'Electricity Consumption', 'Gas Consumption'])

df['Year'] = df['Date'].dt.year
df['Month_Number'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%b')
df['Quarter'] = 'Q' + df['Date'].dt.quarter.astype(str)
df['Year_Month'] = df['Date'].dt.strftime('%Y-%m')
print(df.head(3))


## 2. Dimension Joins (Building Master & Utility Rates)


In [ ]:
bm = df_bm.copy()
bm['Building'] = bm['Building'].astype(str).str.strip()
df = df.merge(bm, on='Building', how='left')

rates_pivot = df_rates.pivot(index='Year', columns='Energy Type', values='Price Per Unit').reset_index()
rates_pivot = rates_pivot.rename(columns={
    'Water': 'Water_Rate',
    'Electricity': 'Electricity_Rate',
    'Gas': 'Gas_Rate'
})
df = df.merge(rates_pivot, on='Year', how='left')
print(df.head(3))


## 3. Financial Cost Engineering
- Water Cost = Water Consumption * Water Rate
- Electricity Cost = Electricity Consumption * Electricity Rate
- Gas Cost = Gas Consumption * Gas Rate
- Total Cost = Water Cost + Electricity Cost + Gas Cost


In [ ]:
df['Water_Cost'] = df['Water_Consumption'] * df['Water_Rate']
df['Electricity_Cost'] = df['Electricity_Consumption'] * df['Electricity_Rate']
df['Gas_Cost'] = df['Gas_Consumption'] * df['Gas_Rate']
df['Total_Cost'] = df['Water_Cost'] + df['Electricity_Cost'] + df['Gas_Cost']

df['Water_Cost_Pct'] = (df['Water_Cost'] / df['Total_Cost']) * 100.0
df['Electricity_Cost_Pct'] = (df['Electricity_Cost'] / df['Total_Cost']) * 100.0
df['Gas_Cost_Pct'] = (df['Gas_Cost'] / df['Total_Cost']) * 100.0

print(f"Total Portfolio Cost: ${df['Total_Cost'].sum():,.2f}")
print(f"Water Cost: ${df['Water_Cost'].sum():,.2f} ({df['Water_Cost'].sum()/df['Total_Cost'].sum()*100:.2f}%)")
print(f"Gas Cost: ${df['Gas_Cost'].sum():,.2f} ({df['Gas_Cost'].sum()/df['Total_Cost'].sum()*100:.2f}%)")
print(f"Electricity Cost: ${df['Electricity_Cost'].sum():,.2f} ({df['Electricity_Cost'].sum()/df['Total_Cost'].sum()*100:.2f}%)")


## 4. Price vs. Volume Variance Decomposition
$$\Delta Cost = (Q_1 - Q_0) \times P_0 + Q_1 \times (P_1 - P_0)$$


In [ ]:
annual_agg = df.groupby('Year').agg({
    'Water_Consumption': 'sum',
    'Electricity_Consumption': 'sum',
    'Gas_Consumption': 'sum',
    'Water_Rate': 'first',
    'Electricity_Rate': 'first',
    'Gas_Rate': 'first',
    'Water_Cost': 'sum',
    'Electricity_Cost': 'sum',
    'Gas_Cost': 'sum',
    'Total_Cost': 'sum'
}).reset_index()

pvd_rows = []
for i in range(1, len(annual_agg)):
    r0 = annual_agg.iloc[i-1]
    r1 = annual_agg.iloc[i]
    period = f"{int(r0['Year'])} -> {int(r1['Year'])}"
    for uname, qcol, pcol, ccol in [('Water', 'Water_Consumption', 'Water_Rate', 'Water_Cost'),
                                    ('Electricity', 'Electricity_Consumption', 'Electricity_Rate', 'Electricity_Cost'),
                                    ('Gas', 'Gas_Consumption', 'Gas_Rate', 'Gas_Cost')]:
        q0, q1 = r0[qcol], r1[qcol]
        p0, p1 = r0[pcol], r1[pcol]
        c0, c1 = r0[ccol], r1[ccol]
        delta_c = c1 - c0
        vol_eff = (q1 - q0) * p0
        prc_eff = q1 * (p1 - p0)
        pvd_rows.append({
            'Period': period, 'Utility': uname,
            'Delta_Cost': round(delta_c, 2),
            'Volume_Effect': round(vol_eff, 2),
            'Price_Effect': round(prc_eff, 2),
            'Vol_Share_Pct': round(vol_eff / delta_c * 100, 2),
            'Price_Share_Pct': round(prc_eff / delta_c * 100, 2)
        })

df_pvd = pd.DataFrame(pvd_rows)
print(df_pvd)


## 5. Export Enriched Dataset


In [ ]:
out_proc = os.path.join("..", "data", "processed")
os.makedirs(out_proc, exist_ok=True)
df.to_csv(os.path.join(out_proc, "energy_consumption_enriched.csv"), index=False)
print("Saved energy_consumption_enriched.csv successfully.")
